In [1]:
import requests

url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
response = requests.get(url)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("Download complete.")

Download complete.


In [2]:
len(text)

1115394

In [3]:
text[:1000]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this in hunger 

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
stoi = { ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s ]
decode = lambda l: ''.join([itos[i] for i in l])

print (encode("hii wassup buddy"))
print (decode(encode("hii")))

[46, 47, 47, 1, 61, 39, 57, 57, 59, 54, 1, 40, 59, 42, 42, 63]
hii


In [6]:
import torch
data = torch.tensor(encode(text), dtype = torch.long)
print(len(data))
data[:100]


1115394


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])

In [7]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
block_size = 8 # block_size = context length
train_data[:block_size +1]  

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [9]:
y = train_data[1:block_size +1]
x = train_data[:block_size]
for t in range(block_size):
    context = x[:t+1]
    target=y[t]
    # print(context.tolist())
    # print("WORDS: ",decode(context), '->', decode([target]),"\tTOKENS: ", context.tolist(), '->', target.item())
    print(f" when input is {context} the target: {target}")


 when input is tensor([18]) the target: 47
 when input is tensor([18, 47]) the target: 56
 when input is tensor([18, 47, 56]) the target: 57
 when input is tensor([18, 47, 56, 57]) the target: 58
 when input is tensor([18, 47, 56, 57, 58]) the target: 1
 when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
 when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
 when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [10]:
torch.manual_seed(1337)
batch_size = 4;block_size = 8
def get_batch(split):
    data = train_data if split =='train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    # print("hello",ix)
    # print([data[i:i+block_size] for i in ix])
    return x,y

xb,yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets: ')
print(yb.shape)
print(yb)

print('--------')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"when input is {context} the target: {target}")



inputs:
torch.Size([4, 8])
tensor([[58, 63,  8,  0,  0, 19, 24, 27],
        [39, 59, 45, 46, 58,  1, 46, 43],
        [49, 43, 57,  1, 53, 50, 42,  1],
        [52, 41, 47, 43, 52, 58,  1, 56]])
targets: 
torch.Size([4, 8])
tensor([[63,  8,  0,  0, 19, 24, 27, 33],
        [59, 45, 46, 58,  1, 46, 43,  1],
        [43, 57,  1, 53, 50, 42,  1, 46],
        [41, 47, 43, 52, 58,  1, 56, 47]])
--------
when input is tensor([58]) the target: 63
when input is tensor([58, 63]) the target: 8
when input is tensor([58, 63,  8]) the target: 0
when input is tensor([58, 63,  8,  0]) the target: 0
when input is tensor([58, 63,  8,  0,  0]) the target: 19
when input is tensor([58, 63,  8,  0,  0, 19]) the target: 24
when input is tensor([58, 63,  8,  0,  0, 19, 24]) the target: 27
when input is tensor([58, 63,  8,  0,  0, 19, 24, 27]) the target: 33
when input is tensor([39]) the target: 59
when input is tensor([39, 59]) the target: 45
when input is tensor([39, 59, 45]) the target: 46
when input is 

In [ ]:
torch.randint(len(data) - block_size, (batch_size,))


tensor([716643, 778003, 937732, 784120])

In [ ]:
print(xb)

tensor([[41, 46,  1, 47, 57,  1, 63, 53],
        [43, 56,  1, 63, 53, 59, 56,  1],
        [63,  1, 46, 47, 57,  1, 42, 43],
        [58,  1, 52, 43, 61, 57,  1, 61]])


In [ ]:
stoi = { ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s ]
decode = lambda l: ''.join([itos[i] for i in l])

print (encode("hii wassup buddy"))
print (decode(encode("hii")))

[46, 47, 47, 1, 61, 39, 57, 57, 59, 54, 1, 40, 59, 42, 42, 63]
hii


In [169]:
x = torch.randn(1,1,65)
# print(x)
x[:,-1, :]

tensor([[-4.1332e-01, -1.2003e+00,  5.7791e-01, -1.2187e-01, -2.0671e-01,
          1.0675e+00,  9.4300e-02, -3.1558e-01,  5.5551e-01,  6.1748e-02,
         -7.2170e-01,  2.5081e-01, -2.9532e-01,  2.1555e-01, -1.6206e+00,
          4.4878e-01,  3.4150e-01,  1.1158e-01, -1.3324e+00,  1.1816e-01,
         -6.1118e-01,  2.1702e-03,  1.9934e-01, -1.5677e+00, -2.7000e-01,
         -1.3783e+00,  1.3452e-01, -1.1704e+00, -1.0062e+00,  3.2306e-01,
          2.8637e-02, -7.7717e-02, -1.0206e+00, -6.9661e-01, -5.4921e-01,
          6.3255e-01,  5.8427e-02, -6.5118e-01,  6.4547e-01, -3.3132e-01,
          5.8529e-01, -1.2204e+00, -4.9620e-01,  1.7178e+00,  8.5121e-01,
         -5.2746e-01,  1.1925e+00, -6.6420e-01,  7.6860e-01, -1.0682e+00,
          4.8913e-01,  6.7704e-01, -2.4156e-01, -4.9996e-01, -6.0669e-01,
          1.8328e+00,  2.9308e-01, -7.1611e-01, -1.9965e-01, -5.6042e-01,
         -1.5364e+00,  2.2658e+00,  1.1407e+00,  8.9348e-01, -2.4000e+00]])

In [198]:
x = torch.randn(65,65)
x[xb].shape

torch.Size([4, 8, 65])

In [234]:
import torch;import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

    def forward(self, idx, targets= None):
        # print(idx.shape)
        logits = self.token_embedding_table(idx) # (B,T,C)
        # print(logits.shape)
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            # print("hi",logits.shape)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss 

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            # print(logits.shape)
            logits = logits[:, -1, :]
            # print(logits.shape)
            probs = F.softmax(logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples = 1)
            # print(idx_next.shape)
            idx = torch.cat((idx,idx_next), dim = 1)
            # print(idx.shape)
        return idx
m = BigramLanguageModel(vocab_size)
print(xb.shape, yb.shape)
output, loss = m(xb,yb)
# print(output.shape, loss.shape)
# ix = m.generate(xb, 8)
print(decode(m.generate(torch.zeros((1,1), dtype = torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 8]) torch.Size([32, 8])

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [222]:
import torch;import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (B,T,C)
        if targets is None:
            loss=None
        else:
            B,T,C = logits.shape
            print(B, T, C)
            logits = logits.view(B*T,C)
            targets =targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        x = 0
        for _ in range(max_new_tokens):
            x =x+1
            logits, loss = self(idx)
            
            logits = logits[:, -1,:]
            if x==1:
                print(logits.shape)
                print(logits) 
                print(decode(torch.argmax(logits,dim=-1).tolist()))
    
            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs,num_samples=1)
            idx = torch.cat((idx, idx_next), dim = 1)
        return idx

m = BigramLanguageModel(vocab_size)
out, loss = m(xb,yb)

print(out.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1,1),dtype=torch.long), max_new_tokens = 100)[0].tolist()))

32 8 65
torch.Size([256, 65])
tensor(4.7830, grad_fn=<NllLossBackward0>)
torch.Size([1, 65])
tensor([[ 0.1808, -0.0700, -0.3596, -0.9152,  0.6258,  0.0255,  0.9545,  0.0643,
          0.3612,  1.1679, -1.3499, -0.5102,  0.2360, -0.2398, -0.9211,  1.5433,
          1.3488, -0.1396,  0.2858,  0.9651, -2.0371,  0.4931,  1.4870,  0.5910,
          0.1260, -1.5627, -1.1601, -0.3348,  0.4478, -0.8016,  1.5236,  2.5086,
         -0.6631, -0.2513,  1.0101,  0.1215,  0.1584,  1.1340, -1.1539, -0.2984,
         -0.5075, -0.9239,  0.5467, -1.4948, -1.2057,  0.5718, -0.5974, -0.6937,
          1.6455, -0.8030,  1.3514, -0.2759, -1.5108,  2.1048,  2.7630, -1.7465,
          1.4516, -1.5103,  0.8212, -0.2115,  0.7789,  1.5333,  1.6097, -0.4032,
         -0.8345]], grad_fn=<SliceBackward0>)
p

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [235]:
optimizer = torch.optim.AdamW(m.parameters(),lr=1e-3)

In [236]:
for name, param in m.named_parameters():
    print(f"Name: {name}")
    print(f"Shape: {param.shape}")
    print(f"Requires grad: {param.requires_grad}")
    # print(f"Values:\n{param.data}")
    print()

Name: token_embedding_table.weight
Shape: torch.Size([65, 65])
Requires grad: True



In [237]:
for name, param in m.named_parameters():
    print(f"{name}: {param.numel()} parameters")
    
total = sum(p.numel() for p in m.parameters())
print(f"\nTotal: {total}")

token_embedding_table.weight: 4225 parameters

Total: 4225


In [238]:
batch_size = 32
epochs = 10000
for steps in range(epochs):
    xb,yb = get_batch('train')
    logits, loss = m(xb,yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.5331473350524902


In [239]:
print(decode(m.generate(idx = torch.zeros((1,1),dtype=torch.long), max_new_tokens = 100)[0].tolist()))


lso br. ave aviu:


Smy, may be ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulsee


# Transformer part

## Mathematical trick in self attention

In [304]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)


In [311]:
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        # print(xprev)
        xbow[b,t] = torch.mean(xprev,0)

Doing above is very hectic and inefficient but we could do something about it with matrix

In [288]:
torch.manual_seed(42)
a = torch.ones(3,3)
b = torch.randint(0, 10, (3,2)).float()
c = a@b
print('a=')
print(a)
print('b=')
print(b)
print('c=')
print(c)


a=
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c=
tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])


above we can see that it kind of added things in a good way but it id like too much so what we will do is

In [298]:
a = torch.tril(torch.ones(3,3))
print(a)
c = a@b
c

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])


tensor([[ 2.,  7.],
        [ 8., 11.],
        [14., 16.]])

now it is adding everything we need to do average and for that apparantely doing the operation as for each row in lower triangualar matrix sums upto 1 we get what we desired and for that we need to do the softmax

but here for understanding in a good manner we will first do the sum way and then the softmax way

In [299]:

a = a/ a.sum(1, keepdim = True)
print(a)
c= a@b
c

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])


tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])

now implementing the above for loop for x

In [321]:
wei = torch.tril(torch.ones(T,T))
wei = wei/ wei.sum(1, keepdim = True)
xbow2 = wei @ x # (T, T) @ (B, T, C) -> (B,T,C)
torch.allclose(xbow, xbow2)

False

better way the softmax way, why do you ask?
well, this makes things better in transformer while the current
 token shdn't see the future tokens. masking concept.

In [324]:
tril= torch.tril(torch.ones(T,T))
wei = torch.ones((T,T))
wei = wei.masked_fill (tril==0, float('-inf'))
wei = F.softmax(wei, dim = -1)
xbow3 = wei@x
torch.allclose(xbow,xbow3)

False

In [310]:
(torch.randn(8,8) @ torch.randn(4,8,2)).shape

torch.Size([4, 8, 2])

In [267]:
x[1,3]

tensor([ 0.6903, -0.1961])

In [268]:
x = torch.tensor([
  # batch 0
  [[ 1.,  2.],
   [ 3.,  4.],
   [ 5.,  6.],
   [ 7.,  8.],
   [ 9., 10.],
   [11., 12.],
   [13., 14.],
   [15., 16.]],

  # batch 1
  [[101.,102.],
   [103.,104.],
   [105.,106.],
   [107.,108.],
   [109.,110.],
   [111.,112.],
   [113.,114.],
   [115.,116.]],

  # batch 2
  [[201.,202.],
   [203.,204.],
   [205.,206.],
   [207.,208.],
   [209.,210.],
   [211.,212.],
   [213.,214.],
   [215.,216.]],

  # batch 3
  [[301.,302.],
   [303.,304.],
   [305.,306.],
   [307.,308.],
   [309.,310.],
   [311.,312.],
   [313.,314.],
   [315.,316.]]
])

In [272]:
x[2,7]

tensor([215., 216.])

In [276]:
xbow

tensor([[[  1.,   2.],
         [  2.,   3.],
         [  3.,   4.],
         [  4.,   5.],
         [  5.,   6.],
         [  6.,   7.],
         [  7.,   8.],
         [  8.,   9.]],

        [[101., 102.],
         [102., 103.],
         [103., 104.],
         [104., 105.],
         [105., 106.],
         [106., 107.],
         [107., 108.],
         [108., 109.]],

        [[201., 202.],
         [202., 203.],
         [203., 204.],
         [204., 205.],
         [205., 206.],
         [206., 207.],
         [207., 208.],
         [208., 209.]],

        [[301., 302.],
         [302., 303.],
         [303., 304.],
         [304., 305.],
         [305., 306.],
         [306., 307.],
         [307., 308.],
         [308., 309.]]])

## Positional Encode

we are doing some code cleanup
1) in embed table we are changing from (vocab_size, vocab_size) to (vocab_size, n_embd)
2) we are adding language model head
3) we need to encode the position as well into the network


In [46]:
import torch;import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)
n_embd = 32
vocab_size = 65
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)#adding this layer
        self.pos_embedding_table = nn.Embedding(vocab_size, n_embd)

    def forward(self, idx, targets= None):
        # print(idx.shape)
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.pos_embedding_table(torch.arange(T, device = device)) # (T,C)
        x = tok_emb+pos_emb  # (B,T,C)
        # logits = self.lm_head(tok_emb) #(B, T, vocab_size)
        logits = self.lm_head(x) #(B, T, vocab_size)
        # print(logits.shape)
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            # print("hi",logits.shape)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss 

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            # print("hi",logits.shape)
            logits = logits[:, -1, :]
            # print(logits.shape)
            probs = F.softmax(logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples = 1)
            # print(idx_next.shape)
            idx = torch.cat((idx,idx_next), dim = 1)
            # print(idx.shape)
        return idx
m = BigramLanguageModel(vocab_size)
print(xb.shape, yb.shape)
output, loss = m(xb,yb)
# print(output.shape, loss.shape)
# ix = m.generate(xb, 8)
print(decode(m.generate(torch.zeros((1,1), dtype = torch.long), max_new_tokens=65)[0].tolist()))

torch.Size([4, 8]) torch.Size([4, 8])

$?3KIOObsMEEFNR:F!SuMyGzRXloNqB& rbUpDjMXM!EjT!vJmfH'NR3cOn.kv$Ac


## Self-Attention part

In [30]:
import torch
torch.manual_seed(1337)
B,T, C = 4, 8, 32
x =torch.randn(B,T,C)
head_size = 16
Q= nn.Linear(C, head_size, bias = False)
K = nn.Linear(C,head_size, bias = False)
value = nn.Linear(C, head_size, bias = False)
q = Q(x)  #(B,T,16)
k = K(x)  # (B,T,16)
wei = q@k.transpose(-2,-1)  #(B, T, 16) @ (B,16,T) -> (B, T, T)

tril= torch.tril(torch.ones(T,T))
# wei = torch.zeros((T,T))
wei = wei.masked_fill (tril==0, float('-inf'))
wei = F.softmax(wei, dim = -1)
v = value(x)
out = wei@v
out.shape

torch.Size([4, 8, 16])

down we can see in each row every position is given equal importance but we don't want that a particular token might need to focus more on one word than another
it is solved with above code.

In [22]:
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

see new wei is like this and again new v is matrix multiplied which is a form of x that we get when we pass it through the linear layer and it is multiplied to wei instead of x

In [29]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5877, 0.4123, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4457, 0.2810, 0.2733, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2220, 0.7496, 0.0175, 0.0109, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0379, 0.0124, 0.0412, 0.0630, 0.8454, 0.0000, 0.0000, 0.0000],
        [0.5497, 0.2187, 0.0185, 0.0239, 0.1831, 0.0062, 0.0000, 0.0000],
        [0.2576, 0.0830, 0.0946, 0.0241, 0.1273, 0.3627, 0.0507, 0.0000],
        [0.0499, 0.1052, 0.0302, 0.0281, 0.1980, 0.2657, 0.1755, 0.1474]],
       grad_fn=<SelectBackward0>)

Some notes from **Andrej** himself:
- Attention is a communication mechanism. Can be seen as nodes in a directed graph looking at each other and aggregating info with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attn simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dim is of course processed completely independently and never "talk" to each other
- In an "encoder" attn block just delete the single that does masking with tril, allowing all tokens to communicate. This block here is called "decode" attn block because it has triangular masking, and is usually used in autoregressive settings, like language modelling.
- self attention just means that keys and values are produced from same source as queries. In cross-attn queries still get produced from x but keys and values comes from some other, external source(eg: encoder module)
- scaled attn additional divides ***wei*** by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and softmax will stay diffuse and not saturate too much. 

Illustration below

In [47]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)
wei = q@k.transpose(-2,-1) *head_size **-0.5
wei1 = q@k.transpose(-2,-1)

In [48]:
k.var(), q.var()

(tensor(0.9101), tensor(1.0700))

In [49]:
wei.var(), wei1.var()

(tensor(1.0852), tensor(17.3625))

as we can see var of wei and wei1 is diff and if we don't do sqrt(head_size) then we will get high variance which will affect our further training coz in previous lecture we found out that the weight if remained gaussian is a good news to achieve stable training.

for eg softmax starts to behave like one hot vector.

To control the variance at intialization

In [54]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim = -1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [56]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim = -1)

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

### Inserting self attention concept to our above code

In [13]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value = nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))


    def forward(self, x):
        B, T, C = x.shape
        key = self.key(x)
        query = self.query(x)
        wei = query@key.transpose(-2,-1) *C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim = -1)
        v = self.value(x)
        out = wei @ v
        return out

In [14]:
block_size

8

In [15]:
import torch;import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)
n_embd = 32
vocab_size = 65
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)#adding this layer
        self.sa_head = Head(n_embd)
        self.pos_embedding_table = nn.Embedding(vocab_size, n_embd)

    def forward(self, idx, targets= None):
        # print(idx.shape)
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.pos_embedding_table(torch.arange(T, device = device)) # (T,C)
        x = tok_emb+pos_emb  # (B,T,C)
        x = self.sa_head(x)
        # logits = self.lm_head(tok_emb) #(B, T, vocab_size)
        logits = self.lm_head(x) #(B, T, vocab_size)
        # print(logits.shape)
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            # print("hi",logits.shape)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss 

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            # print("hi",logits.shape)
            logits = logits[:, -1, :]
            # print(logits.shape)
            probs = F.softmax(logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples = 1)
            # print(idx_next.shape)
            idx = torch.cat((idx,idx_next), dim = 1)
            # print(idx.shape)
        return idx
m = BigramLanguageModel(vocab_size)
print(xb.shape, yb.shape)
output, loss = m(xb,yb)
# print(output.shape, loss.shape)
# ix = m.generate(xb, 8)
print(decode(m.generate(torch.zeros((1,1), dtype = torch.long), max_new_tokens=65)[0].tolist()))

torch.Size([4, 8]) torch.Size([4, 8])

pv,lH ZPDACslAS-nkNch-Nr'w:$jupUj
JI'.G&iuEP !I:alGKL$u.LrCpIadhk


In [96]:
optimizer = torch.optim.AdamW(m.parameters(),lr=1e-3)

In [65]:
batch_size = 32
epochs = 10000
for steps in range(epochs):
    xb,yb = get_batch('train')
    logits, loss = m(xb,yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.381805658340454


In [72]:
print(decode(m.generate(torch.zeros((1,1), dtype = torch.long), max_new_tokens=650)[0].tolist()))


Inthod anofeay thour they meen adyo, tse hastor hbupor mow ogr mewr ave ce te
Ay oil ath met ley as hy hirand whe-ose thers'serer lll nd ma thot pan pary.
Ce
Y her the.
D IONENAGAD:
Busea igd tast
Any ase.
Res in st pha'd I emy st
angond ace, Hederd;
ARime.
Le If no t curie baclow st it ke, om ano thess cardd at nt bs so uthead teat yo kndane rd nds Mononod, thouten an, ncthes the too enjequungt Eno homy at thengthm houl's cil; my bevine alsth?

LAUTES:
'Whous sice han keanaru pt mallsinse agan ds
Wake meinde wind incof toorme hirceegl reang, moupre ut She, hend mee ingew n Iswen bbrepfer acro the ay sevad yon ndet hatheef mins retseen at mte


We got upto 2.38 but we can do better Now, adding Multi-Head Self Attention into our network

In [17]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim = -1)

In [92]:

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)#adding this layer
        self.mha_head = MultiHeadAttention(4, n_embd//4)
        self.pos_embedding_table = nn.Embedding(vocab_size, n_embd)

    def forward(self, idx, targets= None):
        # print(idx.shape)
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.pos_embedding_table(torch.arange(T, device = device)) # (T,C)
        x = tok_emb+pos_emb  # (B,T,C)
        x = self.mha_head(x)
        logits = self.lm_head(x) #(B, T, vocab_size)
        # print(logits.shape)
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            # print("hi",logits.shape)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss 

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            # print("hi",logits.shape)
            logits = logits[:, -1, :]
            # print(logits.shape)
            probs = F.softmax(logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples = 1)
            # print(idx_next.shape)
            idx = torch.cat((idx,idx_next), dim = 1)
            # print(idx.shape)
        return idx
m = BigramLanguageModel(vocab_size)
print(xb.shape, yb.shape)
output, loss = m(xb,yb)
# print(output.shape, loss.shape)
# ix = m.generate(xb, 8)
print(decode(m.generate(torch.zeros((1,1), dtype = torch.long), max_new_tokens=65)[0].tolist()))

torch.Size([4, 8]) torch.Size([4, 8])

nt&Z:MXNB;Aqx;Q'lNLaf'Lpwu. hs;SyklR$NE!yDEzs
!.YJhQJ seJpAnGYbfi


In [108]:
m = BigramLanguageModel(vocab_size)  # ✅ New model
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)  # ✅ Must create AFTER model

In [99]:
batch_size = 32
epochs = 10000
for steps in range(epochs):
    xb,yb = get_batch('train')
    logits, loss = m(xb,yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    # print(loss.item())
print(loss.item())

2.1716837882995605


In [100]:
print(decode(m.generate(torch.zeros((1,1), dtype = torch.long), max_new_tokens=650)[0].tolist()))


Mowsh.

OFor bet,
Wiken le wir come lledingefich's sore. By Pigeamad, nowhend: Gt
Thor may. dive nase ans me paf of yo. You the tith my your,
We shea, ansich an nott ords,
I fanty tre'd to sord, thee you as do cear, Whacse
Add ow, enied not gre abet buar;
Falllvew!

MING HER: and hans fas swair'd fore ave now sen comor tef ene gheix brient cor ce moun therebleardew had, shord; he anche igo, tathim'd the sram nelito my than its's wer thatle.

Hpll daptatri! st you bre!

S Jow lld ave soms grais of marg,
Ped fe withagester ey mo? spato no clal is will graldiecetcamy? you lad;
And me thes!

CLES:
tor gon s I bush warmanly he pris sord hous:
My g


adding MHA got upto 2.17 which is a good news and meaning we are hearning to the correct direction. So, we can do better. Now adding FFN

In [18]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU()

        )

    def forward(self, x):
        return self.net(x)

In [19]:

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)#adding this layer
        self.mha_head = MultiHeadAttention(4, n_embd//4)
        self.ffwd =FeedForward(n_embd)
        self.pos_embedding_table = nn.Embedding(vocab_size, n_embd)

    def forward(self, idx, targets= None):
        # print(idx.shape)
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.pos_embedding_table(torch.arange(T, device = device)) # (T,C)
        x = tok_emb+pos_emb  # (B,T,C)
        x = self.mha_head(x)
        x = self.ffwd(x)
        logits = self.lm_head(x) #(B, T, vocab_size)
        # print(logits.shape)
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            # print("hi",logits.shape)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss 

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            # print("hi",logits.shape)
            logits = logits[:, -1, :]
            # print(logits.shape)
            probs = F.softmax(logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples = 1)
            # print(idx_next.shape)
            idx = torch.cat((idx,idx_next), dim = 1)
            # print(idx.shape)
        return idx
m = BigramLanguageModel(vocab_size)
print(xb.shape, yb.shape)
output, loss = m(xb,yb)
# print(output.shape, loss.shape)
# ix = m.generate(xb, 8)
print(decode(m.generate(torch.zeros((1,1), dtype = torch.long), max_new_tokens=65)[0].tolist()))

torch.Size([4, 8]) torch.Size([4, 8])

jfWQ'N 'vjVARBE?ND;x'gb;-RZO!V$DE?vTWyJFVdalJ&vWB-3;vAEPj3AI.suLz


In [110]:
batch_size = 32
epochs = 10000
for steps in range(epochs):
    xb,yb = get_batch('train')
    logits, loss = m(xb,yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    # print(loss.item())
print(loss.item())

2.083311080932617


In [112]:
print(decode(m.generate(torch.zeros((1,1), dtype = torch.long), max_new_tokens=225)[0].tolist()))


Dutt ind upted, que aplid gonen, hand les's he tish's to whannowis onee?
Har souns oosld mais sto ale dolsy, thy mree hel aing chiccought eon ous, mempous.

But-rutny to
no hear bee'st, agentap onlior noter it thers
Lemss ler


Now, again adding this again reduced it to 2.08 but still there are somethings more
1) Adding the block of transformer layer like what we did we need to replicate it to Nx times 
and
2) As the network becomes deeper we need to do some better optimization with Layer Norm and Residual connection.

In [21]:
import torch;import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)
n_embd = 32
vocab_size = 65
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value = nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))


    def forward(self, x):
        B, T, C = x.shape
        key = self.key(x)
        query = self.query(x)
        wei = query@key.transpose(-2,-1) *C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim = -1)
        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out
        # return torch.cat([h(x) for h in self.heads], dim = -1)


class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4* n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),


        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        head_size = n_embd//n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) #residual conxn implemented
        x = x+ self.ffwd(self.ln2(x))  #residual conxn implemented
        return x
    

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
        self.pos_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.Sequential(
                Block(n_embd, n_head = 4),
                Block(n_embd, n_head = 4),
                Block(n_embd, n_head = 4),
                nn.LayerNorm(n_embd),
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)#adding this layer
        # self.mha_head = MultiHeadAttention(4, n_embd//4)
        # self.ffwd =FeedForward(n_embd)
        

    def forward(self, idx, targets= None):
        # print(idx.shape)
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.pos_embedding_table(torch.arange(T, device = device)) # (T,C)
        x = tok_emb+pos_emb  # (B,T,C)
        x = self.blocks(x)
        logits = self.lm_head(x) #(B, T, vocab_size)
        # print(logits.shape)
        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            # print("hi",logits.shape)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss 

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            # print("hi",logits.shape)
            logits = logits[:, -1, :]
            # print(logits.shape)
            probs = F.softmax(logits, dim = -1)
            idx_next = torch.multinomial(probs, num_samples = 1)
            # print(idx_next.shape)
            idx = torch.cat((idx,idx_next), dim = 1)
            # print(idx.shape)
        return idx
m = BigramLanguageModel(vocab_size)
print(xb.shape, yb.shape)
output, loss = m(xb,yb)
# print(output.shape, loss.shape)
# ix = m.generate(xb, 8)
print(decode(m.generate(torch.zeros((1,1), dtype = torch.long), max_new_tokens=65)[0].tolist()))

torch.Size([32, 8]) torch.Size([32, 8])

. 'bHp-qeCfOZgqrviqEQ$
OZS-ds'iHK LmWVXuZR$zQIk N
v:?jorTy'FwvyEf


In [ ]:
#without layernorm

m = BigramLanguageModel(vocab_size)  # ✅ New model
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)  # ✅ Must create AFTER model
batch_size = 32
epochs = 10000
for steps in range(epochs):
    xb,yb = get_batch('train')
    logits, loss = m(xb,yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    # print(loss.item())
print(loss.item())
print(decode(m.generate(torch.zeros((1,1), dtype = torch.long), max_new_tokens=65)[0].tolist()))

1.9061615467071533

SICINIUS:
Prager he tell hat ten, entuse, there from him hus farr


In [16]:
print(decode(m.generate(torch.zeros((1,1), dtype = torch.long), max_new_tokens=650)[0].tolist()))


You no gone is my lettal!
And us my bolle-tay this.
I hand leore starry him hold, him they for hearduings that just rain a the ping hant and with inster
They you.' but Sain thy best thee innate porrook
For up exwilto are offaius twee heady? rever shome me heaven
That in damain'd:
To like is suerful, that appome a to main a very so shall blood old's the will in weet the
That but holy armings.

COLLO:
So unly more with in heold
have itents and age heard so feasends of 'swenteld Bure theirs a this he son falt full.

SIGWARD ICIUS:
And rither and shalmer.' han one reile?

HESS RICIII:
In a thee do cannwin in you,
And poivil sink hattery shay blug


Now, We add the layernorm as well which normalises across features like BN which did across Batch. and we are doing it across 1 input which we will always have we actually don't need that buffer of running_mean and std

In [ ]:
class LayerNorm:
    def __init__(self, dim,epsilon = 1e-5):
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        self.eps = epsilon

    def __call__(self, x):
        
        x_mean  = x.mean(1, keepdim = True) #change 0 to 1
        x_std = x.var(1, keepdim = True ) #change 0 to 1
        
        x_mean = self.running_mean
        x_std = self.running_std

        self.out = self.gamma * ((x - x_mean)/torch.sqrt(x_std+ self.eps)) + self.beta
        return self.out
    
    def parameters(self):
        return [self.gamma, self.beta]
    

In [23]:
#with layernorm

m = BigramLanguageModel(vocab_size)  # ✅ New model
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)  # ✅ Must create AFTER model
batch_size = 32
epochs = 10000
for steps in range(epochs):
    xb,yb = get_batch('train')
    logits, loss = m(xb,yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    # print(loss.item())
print(loss.item())
print(decode(m.generate(torch.zeros((1,1), dtype = torch.long), max_new_tokens=650)[0].tolist()))

1.9491214752197266

CLARENCES:
Aw; thou hem, we a henour,
To sool, and soul that arged.
 Clavinstit's noe.

ESCAMINGHAS:
By fid sive
Recants thinSen a minty in Worth,
That ast not more thou hapse not prace bid on!I thrioor we did shrun---gatentys unters apnoin to it thy cust a dride;
Their to othis opniont, her, not fay.

LORCKINGS:
Dest hends,
Ye such would never: exice of thy bation to und and my no;
And the swear, if ot thrior thmines a king to vastenity:
he so here was Thou.

RUCIUMR:
A town deam untel,
Denery, sareice
Here the was, and
The with hels in thy bely.
Near. Loving the remise: upere by tyroven Thatmy unt price I maide.
Be, monts hopted nobly thy g


## Full Training only 1 cell

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from tqdm import trange, tqdm


# hyperparameters
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 256 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in trange(eval_iters, desc=f"Eval {split}", leave=False):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # better init, not covered in the original GPT video, but important, will cover in followup video
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = GPTLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in trange(max_iters, desc="Training", unit="step"):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        tqdm.write(
            f"step {iter}: train loss {losses['train']:.4f}, "
            f"val loss {losses['val']:.4f}"
        )

    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))
# open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

10.788929 M parameters


Training:   0%|          | 0/5000 [28:07<?, ?step/s]

step 0: train loss 4.2221, val loss 4.2306


Training:   6%|▌         | 283/5000 [1:49:56<33:09:16, 25.30s/step]